<a href="https://colab.research.google.com/github/tevfikaytekin/recommender_systems_course/blob/main/gnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. Download and Prepare the Gowalla Dataset
Run this cell to download the dataset from Stanford SNAP and save the user-item interactions to `dataset/gowalla/gowalla.inter`.

In [1]:
import os
import pandas as pd

# Create the dataset directory
os.makedirs('dataset/gowalla', exist_ok=True)

# Download the raw dataset
print("Downloading Gowalla dataset from SNAP...")
!wget -nc -q --show-progress https://snap.stanford.edu/data/loc-gowalla_totalCheckins.txt.gz

# Extract the gzip file
print("Extracting dataset...")
!gunzip -f loc-gowalla_totalCheckins.txt.gz

# Read and convert to a simpler format
print("Converting to interaction format...")
df = pd.read_csv(
    'loc-gowalla_totalCheckins.txt',
    sep='\t',
    header=None,
    names=['user', 'checkin_time', 'latitude', 'longitude', 'item']
)

# We only need user and item for collaborative filtering
df_inter = df[['user', 'item']]
df_inter.to_csv('dataset/gowalla/gowalla.inter', sep='\t', index=False)

print("Dataset successfully prepared and saved to dataset/gowalla/gowalla.inter!")

loc-gowalla_totalCh 100%[===================>] 100.58M  25.5MB/s    in 5.5s    
Extracting dataset...
Converting to interaction format...
Dataset successfully prepared and saved to dataset/gowalla/gowalla.inter!


In [ ]:
df_inter.head()

,user,item
0,0,22847
1,0,420315
2,0,316637
3,0,16516
4,0,5535878


### 1. Data Preprocessing
For educational purposes and to keep training time reasonable on Colab, we will filter the Gowalla dataset to keep only users and items with a minimum number of interactions (e.g., 50-core). Then, we will split the data into training and testing sets and create PyTorch data loaders.

In [2]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
import scipy.sparse as sp

# Load the processed inter file
df = pd.read_csv('dataset/gowalla/gowalla.inter', sep='\t')
df.columns = ['user', 'item']

# Filter 50-core for educational speed
user_counts = df['user'].value_counts()
item_counts = df['item'].value_counts()
df = df[df['user'].isin(user_counts[user_counts >= 50].index)]
df = df[df['item'].isin(item_counts[item_counts >= 50].index)]

# Map IDs to contiguous indices
user_mapping = {u: i for i, u in enumerate(df['user'].unique())}
item_mapping = {i: j for j, i in enumerate(df['item'].unique())}
df['user'] = df['user'].map(user_mapping)
df['item'] = df['item'].map(item_mapping)

num_users = len(user_mapping)
num_items = len(item_mapping)
print(f"Users: {num_users}, Items: {num_items}, Interactions: {len(df)}")

# Train-Test Split (Leave-one-out or random 80/20)
# For simplicity, we do a random 80/20 split per user
train_data = []
test_data = defaultdict(list)

grouped = df.groupby('user')
for user, group in grouped:
    items = group['item'].tolist()
    np.random.shuffle(items)
    split_idx = int(len(items) * 0.8)
    train_data.extend([(user, item) for item in items[:split_idx]])
    test_data[user] = items[split_idx:]

#train_df = pd.DataFrame(train_data, columns=['user', 'item'])

# Build adjacency matrix for GNN
R = sp.dok_matrix((num_users, num_items), dtype=np.float32)
for u, i in train_data:
    R[u, i] = 1.0
R = R.tolil()

print(f"Train interactions: {len(train_data)}")
print(f"Test users: {len(test_data)}")

Users: 30281, Items: 12048, Interactions: 1162837
Train interactions: 917516
Test users: 30281


In [3]:
train_data[:5]

[(0, 53), (0, 81), (0, 49), (0, 60), (0, 27)]

In [4]:
# test_data is a dictionary mapping user -> [test items]
# Let's preview the test items for the first 5 users
print("Preview of test_data (first 5 users):")
for i, (user, items) in enumerate(test_data.items()):
    if i >= 5: break
    print(f"User {user}: {items[:5]} {'...' if len(items) > 5 else ''}")

Preview of test_data (first 5 users):
User 0: [32, 27, 1, 1, 46] ...
User 1: [161, 111, 124, 109, 140] ...
User 2: [82, 170, 164, 162, 82] ...
User 3: [194, 8, 192, 206, 198] ...
User 4: [230, 222, 221, 6, 2] ...


### 2. Evaluation Metrics From Scratch
We implement Precision@K, Recall@K, and NDCG@K.

In [5]:
import math

def recall_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    pred_k = set(predicted[:k])
    actual_set = set(actual)
    return len(pred_k & actual_set) / len(actual_set)

def precision_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    pred_k = set(predicted[:k])
    actual_set = set(actual)
    return len(pred_k & actual_set) / k

def ndcg_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    pred_k = predicted[:k]
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(actual), k)))
    dcg = sum(1.0 / math.log2(i + 2) for i, p in enumerate(pred_k) if p in actual)
    return dcg / idcg

def evaluate_model(model, test_data, train_matrix, k=20, device='cuda'):
    model.eval()
    precisions, recalls, ndcgs = [], [], []

    with torch.no_grad():
        # Precompute embeddings ONCE for all users and items to speed up evaluation
        users_emb, items_emb = model.get_embeddings()

        for user, actual_items in test_data.items():
            u_emb = users_emb[user]
            # Fast dot product for all items for this specific user
            scores = torch.matmul(u_emb, items_emb.T)

            # Mask training items
            train_items = train_matrix.rows[user]
            scores[train_items] = -float('inf')

            _, top_k_items = torch.topk(scores, k)
            top_k_items = top_k_items.cpu().tolist()

            precisions.append(precision_at_k(actual_items, top_k_items, k))
            recalls.append(recall_at_k(actual_items, top_k_items, k))
            ndcgs.append(ndcg_at_k(actual_items, top_k_items, k))

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

### 3. Matrix Factorization (MF) from Scratch
A standard BPR Matrix Factorization model using PyTorch `nn.Embedding`.

In [6]:
import torch.nn as nn

class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(MatrixFactorization, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, users, pos_items, neg_items):
        u_emb = self.user_emb(users)
        pos_i_emb = self.item_emb(pos_items)
        neg_i_emb = self.item_emb(neg_items)

        pos_scores = (u_emb * pos_i_emb).sum(dim=1)
        neg_scores = (u_emb * neg_i_emb).sum(dim=1)

        return pos_scores, neg_scores

    def get_embeddings(self):
        return self.user_emb.weight, self.item_emb.weight

### 4. Graph Neural Network (LightGCN) from Scratch
We implement a simplified LightGCN that propagates embeddings over the user-item bipartite graph.

In [7]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, adj_matrix, embedding_dim=64, n_layers=2):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.n_layers = n_layers

        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

        self.norm_adj = self._build_norm_adj(adj_matrix)

    def _build_norm_adj(self, R):
        # Create A = [0, R; R^T, 0]
        adj_mat = sp.dok_matrix((self.num_users + self.num_items, self.num_users + self.num_items), dtype=np.float32)
        adj_mat = adj_mat.tolil()

        # R is already converted to LIL in the preprocessing step
        adj_mat[:self.num_users, self.num_users:] = R
        adj_mat[self.num_users:, :self.num_users] = R.T
        adj_mat = adj_mat.todok()

        # Normalize A
        rowsum = np.array(adj_mat.sum(axis=1))

        # Safely ignore the divide by zero warning for isolated nodes
        with np.errstate(divide='ignore'):
            d_inv = np.power(rowsum, -0.5).flatten()
        d_inv[np.isinf(d_inv)] = 0.

        d_mat = sp.diags(d_inv)
        norm_adj = d_mat.dot(adj_mat).dot(d_mat).tocoo()

        # Convert to sparse PyTorch tensor
        # Fixed the performance warning by converting to np.array first
        indices = torch.LongTensor(np.array([norm_adj.row, norm_adj.col]))
        values = torch.FloatTensor(norm_adj.data)
        sparse_adj = torch.sparse_coo_tensor(indices, values, torch.Size(norm_adj.shape))
        return sparse_adj

    def forward(self, users, pos_items, neg_items):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight])
        embs = [all_emb]

        # Message passing layers
        g_droped_adj = self.norm_adj.to(all_emb.device)
        for layer in range(self.n_layers):
            all_emb = torch.sparse.mm(g_droped_adj, all_emb)
            embs.append(all_emb)

        embs = torch.stack(embs, dim=1)
        light_out = torch.mean(embs, dim=1)

        users_emb, items_emb = torch.split(light_out, [self.num_users, self.num_items])

        u_emb = users_emb[users]
        pos_i_emb = items_emb[pos_items]
        neg_i_emb = items_emb[neg_items]

        pos_scores = (u_emb * pos_i_emb).sum(dim=1)
        neg_scores = (u_emb * neg_i_emb).sum(dim=1)

        return pos_scores, neg_scores

    def get_embeddings(self):
        # Return precomputed embeddings for fast evaluation
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight])
        embs = [all_emb]
        g_droped_adj = self.norm_adj.to(all_emb.device)
        for layer in range(self.n_layers):
            all_emb = torch.sparse.mm(g_droped_adj, all_emb)
            embs.append(all_emb)
        embs = torch.stack(embs, dim=1)
        light_out = torch.mean(embs, dim=1)
        users_emb, items_emb = torch.split(light_out, [self.num_users, self.num_items])
        return users_emb, items_emb

### Mathematical Note: Graph Normalization (`d_inv`)
In the `_build_norm_adj` method, `d_inv` represents the **inverse square root of the degree matrix** ($D^{-1/2}$).

**Why normalize?**
In a GNN, nodes aggregate information from their neighbors. If a user or item has many interactions (high degree), their embedding values would grow significantly larger than others during aggregation, leading to numerical instability and poor training.

**Symmetric Normalization:**
We apply symmetric normalization using the formula:
$$\hat{A} = D^{-1/2} A D^{-1/2}$$

Where:
* $A$ is the adjacency matrix.
* $D$ is the diagonal degree matrix, where $D_{ii} = \sum_j A_{ij}$.
* The operation scales the message from node $i$ to $j$ by $\frac{1}{\sqrt{\text{deg}(i) \times \text{deg}(j)}}$, ensuring the embeddings remain at a consistent scale across the graph.

In [8]:
import numpy as np

# 1. Create a 4-node graph to show different degree effects
# Node 0 (Hub): connected to 1, 2, 3 (degree 3)
# Node 1: connected to 0, 2 (degree 2)
# Node 2: connected to 0, 1 (degree 2)
# Node 3 (Isolated): connected only to 0 (degree 1)
A = np.array([
    [0, 1, 1, 1],
    [1, 0, 1, 0],
    [1, 1, 0, 0],
    [1, 0, 0, 0]
], dtype=float)

print("Original Adjacency Matrix (A):\n", A)

# 2. Calculate the degree of each node
degrees = A.sum(axis=1)
print("\nNode Degrees:", degrees)

# 3. Calculate D^{-1/2} (d_inv)
d_inv = np.power(degrees, -0.5)
print("\nInverse Square Root of Degrees (d_inv):\n", np.round(d_inv, 3))

# 4. Create the diagonal matrix D^{-1/2}
D_inv_mat = np.diag(d_inv)

# 5. Calculate Symmetric Normalization: D^{-1/2} * A * D^{-1/2}
norm_A = D_inv_mat.dot(A).dot(D_inv_mat)
print("\nSymmetrically Normalized Adjacency Matrix (\u00c2):\n", np.round(norm_A, 3))

print("\n--- OBSERVATIONS ---")
print("Edge (0,1): Hub (deg 3) <-> Node (deg 2) | Weight = 1/sqrt(3*2) = 0.408 (Heavily penalized)")
print("Edge (1,2): Node (deg 2) <-> Node (deg 2)| Weight = 1/sqrt(2*2) = 0.500 (Moderately penalized)")
print("Edge (0,3): Hub (deg 3) <-> Node (deg 1) | Weight = 1/sqrt(3*1) = 0.577 (Less penalized since Node 3 relies completely on Node 0)")

Original Adjacency Matrix (A):
 [[0. 1. 1. 1.]
 [1. 0. 1. 0.]
 [1. 1. 0. 0.]
 [1. 0. 0. 0.]]

Node Degrees: [3. 2. 2. 1.]

Inverse Square Root of Degrees (d_inv):
 [0.577 0.707 0.707 1.   ]

Symmetrically Normalized Adjacency Matrix (Â):
 [[0.    0.408 0.408 0.577]
 [0.408 0.    0.5   0.   ]
 [0.408 0.5   0.    0.   ]
 [0.577 0.    0.    0.   ]]

--- OBSERVATIONS ---
Edge (0,1): Hub (deg 3) <-> Node (deg 2) | Weight = 1/sqrt(3*2) = 0.408 (Heavily penalized)
Edge (1,2): Node (deg 2) <-> Node (deg 2)| Weight = 1/sqrt(2*2) = 0.500 (Moderately penalized)
Edge (0,3): Hub (deg 3) <-> Node (deg 1) | Weight = 1/sqrt(3*1) = 0.577 (Less penalized since Node 3 relies completely on Node 0)


### Note: Are Graphs in GNNs Directed or Undirected?
In a Graph Neural Network (GNN), the graph can be **either directed or undirected**, depending on the specific problem and the nature of the data you are modeling.

* **Undirected Graphs:** The edges have no direction, meaning the relationship between two nodes is symmetric. For example, in a social network where user A and user B are "friends," the connection goes both ways. In our **LightGCN** model, the Gowalla user-item bipartite graph is treated as undirected (if a user interacted with an item, we add edges in both directions), which is why we apply *symmetric* normalization.
* **Directed Graphs:** The edges have a specific direction, indicating an asymmetric relationship. For example, on Twitter, User A follows User B, but B might not follow A. In GNNs dealing with directed graphs, message passing often needs to be specialized (e.g., aggregating only from incoming edges).

### Mathematical Note: Neighborhood Aggregation (Message Passing)
This simple example demonstrates what happens under the hood when we call `torch.sparse.mm(adj_matrix, all_emb)` in a Graph Neural Network.

In [9]:
import torch

# 1. Create a tiny 3-node graph Adjacency Matrix
# Node 0 is connected to Node 1 and Node 2
# Node 1 is connected to Node 0
# Node 2 is connected to Node 0
indices = torch.tensor([
    [0, 0, 1, 2], # row indices
    [1, 2, 0, 0]  # col indices
])
values = torch.tensor([1.0, 1.0, 1.0, 1.0])
adj_matrix = torch.sparse_coo_tensor(indices, values, (3, 3))

print("Sparse Adjacency Matrix (A) as Dense:")
print(adj_matrix.to_dense())

# 2. Create Toy Embeddings for the 3 nodes (e.g., 2-dimensional features)
all_emb = torch.tensor([
    [10.0, 10.0], # Node 0 embeddings
    [1.0,  2.0],  # Node 1 embeddings
    [3.0,  4.0]   # Node 2 embeddings
])

print("\nOriginal Embeddings (all_emb):")
print(all_emb)

# 3. Perform the Message Passing operation!
new_emb = torch.sparse.mm(adj_matrix, all_emb)

print("\nNew Embeddings after torch.sparse.mm (Aggregated):")
print(new_emb)

print("\n--- Why did this happen? ---")
print("Node 0's new embedding is [1+3, 2+4] = [4, 6] (Sum of its neighbors: Node 1 and 2)")
print("Node 1's new embedding is [10, 10] (Sum of its neighbor: Node 0)")
print("Node 2's new embedding is [10, 10] (Sum of its neighbor: Node 0)")
print("This simple addition is the foundation of Message Passing! \nWhen we add the normalization step we discussed earlier, instead of simple addition, \nit becomes a weighted average.")

Sparse Adjacency Matrix (A) as Dense:
tensor([[0., 1., 1.],
        [1., 0., 0.],
        [1., 0., 0.]])

Original Embeddings (all_emb):
tensor([[10., 10.],
        [ 1.,  2.],
        [ 3.,  4.]])

New Embeddings after torch.sparse.mm (Aggregated):
tensor([[ 4.,  6.],
        [10., 10.],
        [10., 10.]])

--- Why did this happen? ---
Node 0's new embedding is [1+3, 2+4] = [4, 6] (Sum of its neighbors: Node 1 and 2)
Node 1's new embedding is [10, 10] (Sum of its neighbor: Node 0)
Node 2's new embedding is [10, 10] (Sum of its neighbor: Node 0)
This simple addition is the foundation of Message Passing! 
When we add the normalization step we discussed earlier, instead of simple addition, 
it becomes a weighted average.


/tmp/ipykernel_970/2227093882.py:12: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj_matrix = torch.sparse_coo_tensor(indices, values, (3, 3))


### 5. Training Loop & Comparison
We define a BPR loss function, a PyTorch DataLoader, and train both models to compare them.

In [10]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import time

class BPRDataset(Dataset):
    def __init__(self, train_data, num_items):
        self.train_data = train_data
        self.num_items = num_items

    def __len__(self):
        return len(self.train_data)

    def __getitem__(self, idx):
        user, pos_item = self.train_data[idx]
        # Negative sampling
        neg_item = np.random.randint(0, self.num_items)
        return user, pos_item, neg_item

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bpr_dataset = BPRDataset(train_data, num_items)
dataloader = DataLoader(bpr_dataset, batch_size=4096, shuffle=True)

def train_model(model, epochs=5, lr=0.001):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"Training {model.__class__.__name__}...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        start_time = time.time()
        for users, pos_items, neg_items in dataloader:
            users, pos_items, neg_items = users.to(device), pos_items.to(device), neg_items.to(device)

            optimizer.zero_grad()
            pos_scores, neg_scores = model(users, pos_items, neg_items)

            # BPR Loss = -log(sigmoid(pos_score - neg_score))
            loss = -F.logsigmoid(pos_scores - neg_scores).mean()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f} | Time: {time.time()-start_time:.2f}s")

    # Evaluate
    print(f"Evaluating {model.__class__.__name__}...")
    p, r, n = evaluate_model(model, test_data, R, k=20, device=device)
    print(f"Precision@20: {p:.4f} | Recall@20: {r:.4f} | NDCG@20: {n:.4f}\n")
    return p, r, n



In [11]:
# 1. Train standard MF
mf_model = MatrixFactorization(num_users, num_items, embedding_dim=64)
mf_metrics = train_model(mf_model, epochs=10)

# 2. Train GNN
gnn_model = LightGCN(num_users, num_items, R, embedding_dim=64, n_layers=2)
gnn_metrics = train_model(gnn_model, epochs=10)

Training MatrixFactorization...
Epoch 1/10 | Loss: 0.6470 | Time: 5.51s
Epoch 2/10 | Loss: 0.5464 | Time: 4.61s
Epoch 3/10 | Loss: 0.4567 | Time: 4.54s
Epoch 4/10 | Loss: 0.3696 | Time: 4.65s
Epoch 5/10 | Loss: 0.2864 | Time: 4.57s
Epoch 6/10 | Loss: 0.2184 | Time: 4.61s
Epoch 7/10 | Loss: 0.1676 | Time: 4.54s
Epoch 8/10 | Loss: 0.1315 | Time: 4.60s
Epoch 9/10 | Loss: 0.1050 | Time: 4.60s
Epoch 10/10 | Loss: 0.0868 | Time: 4.62s
Evaluating MatrixFactorization...
Precision@20: 0.0191 | Recall@20: 0.0905 | NDCG@20: 0.0566

Training LightGCN...
Epoch 1/10 | Loss: 0.5847 | Time: 6.97s
Epoch 2/10 | Loss: 0.2593 | Time: 6.64s
Epoch 3/10 | Loss: 0.1545 | Time: 6.74s
Epoch 4/10 | Loss: 0.1130 | Time: 6.65s
Epoch 5/10 | Loss: 0.0920 | Time: 6.69s
Epoch 6/10 | Loss: 0.0783 | Time: 6.63s
Epoch 7/10 | Loss: 0.0693 | Time: 6.70s
Epoch 8/10 | Loss: 0.0637 | Time: 6.69s
Epoch 9/10 | Loss: 0.0585 | Time: 6.69s
Epoch 10/10 | Loss: 0.0549 | Time: 6.68s
Evaluating LightGCN...
Precision@20: 0.0229 | Recal